In [1]:
from pathlib import Path
import polars as pl

DATA_DIR = Path(
    r"C:\Users\raihan bhakti\id_reg_kg_2511"
)

parquet_files = sorted(DATA_DIR.glob("*.parquet"))

print("Versi Polars:", pl.__version__)
print("Jumlah file Parquet:", len(parquet_files))

lf = pl.scan_parquet(
    [str(f) for f in parquet_files]
)

print("\nSchema dataset:")
print(lf.collect_schema())

Versi Polars: 1.44.2
Jumlah file Parquet: 75

Schema dataset:
Schema([('global_id', Int64), ('local_id', Int64), ('regulation_type', String), ('enacting_body', String), ('regulation_number', String), ('year', String), ('about', String), ('effective_date', String), ('chapter', String), ('article', String), ('content', String), ('chunk_id', Int64), ('embedding', List(Float64)), ('tfidf_vector', List(Float64)), ('__index_level_0__', Int64), ('kg_entities_json', String), ('kg_entity_count', Int64), ('kg_cross_references_json', String), ('kg_cross_ref_count', Int64), ('kg_legal_domains_json', String), ('kg_primary_domain', String), ('kg_domain_confidence', Float64), ('kg_concept_clusters_json', String), ('kg_cluster_count', Int64), ('kg_cluster_diversity', Float64), ('kg_authority_score', Float64), ('kg_hierarchy_level', Int64), ('kg_temporal_score', Float64), ('kg_years_old', Int64), ('kg_legal_richness', Float64), ('kg_legal_complexity', Float64), ('kg_legal_actions_json', String), ('kg_a

In [2]:

# Menghitung jumlah baris seluruh dataset
total_baris = (
    lf.select(pl.len().alias("total_baris"))
    .collect(engine="streaming")
)

print("Jumlah baris dataset:")
print(total_baris)


Jumlah baris dataset:
shape: (1, 1)
┌─────────────┐
│ total_baris │
│ ---         │
│ u32         │
╞═════════════╡
│ 748558      │
└─────────────┘


In [3]:

kolom_preview = [
    "global_id",
    "regulation_type",
    "enacting_body",
    "regulation_number",
    "year",
    "effective_date",
    "about",
    "content",
]

sampel = (
    lf.select(kolom_preview)
    .head(5)
    .collect(engine="streaming")
)

display(sampel)


global_id,regulation_type,enacting_body,regulation_number,year,effective_date,about,content
i64,str,str,str,str,str,str,str
0,"""PERATURAN PEMERINTAH""","""REPUBLIK INDONESIA""","""26""","""2009""","""3 Maret 2009""","""TATA CARA PENGENAAN SANKSI ADM…","""Menimbang : Mengingat : Meneta…"
1,"""PERATURAN PEMERINTAH""","""REPUBLIK INDONESIA""","""26""","""2009""","""3 Maret 2009""","""TATA CARA PENGENAAN SANKSI ADM…","""Dalam Peraturan Pemerintah ini…"
2,"""PERATURAN PEMERINTAH""","""REPUBLIK INDONESIA""","""26""","""2009""","""3 Maret 2009""","""TATA CARA PENGENAAN SANKSI ADM…","""- (1) Sanksi administrasi beru…"
3,"""PERATURAN PEMERINTAH""","""REPUBLIK INDONESIA""","""26""","""2009""","""3 Maret 2009""","""TATA CARA PENGENAAN SANKSI ADM…","""- (1) Besarnya sanksi administ…"
4,"""PERATURAN PEMERINTAH""","""REPUBLIK INDONESIA""","""26""","""2009""","""3 Maret 2009""","""TATA CARA PENGENAAN SANKSI ADM…","""- (1) Besarnya sanksi administ…"


In [4]:

# Audit nilai kosong pada kolom utama

schema = lf.collect_schema()

kolom_teks = [
    "regulation_type",
    "enacting_body",
    "regulation_number",
    "year",
    "effective_date",
    "chapter",
    "article",
    "about",
    "content",
]

kolom_teks = [
    c for c in kolom_teks
    if c in schema
]

kolom_audit = [
    "global_id",
    *kolom_teks,
]

ekspresi = [
    pl.len().alias("total_baris")
]

for kolom in kolom_audit:
    ekspresi.append(
        pl.col(kolom).null_count().alias(
            f"{kolom}_null"
        )
    )

    if kolom in kolom_teks:
        ekspresi.append(
            pl.col(kolom)
            .str.strip_chars()
            .eq("")
            .fill_null(False)
            .sum()
            .alias(f"{kolom}_string_kosong")
        )

hasil_audit = (
    lf.select(ekspresi)
    .collect(engine="streaming")
)

display(hasil_audit)


total_baris,global_id_null,regulation_type_null,regulation_type_string_kosong,enacting_body_null,enacting_body_string_kosong,regulation_number_null,regulation_number_string_kosong,year_null,year_string_kosong,effective_date_null,effective_date_string_kosong,chapter_null,chapter_string_kosong,article_null,article_string_kosong,about_null,about_string_kosong,content_null,content_string_kosong
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
748558,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:

distribusi_tahun = (
    lf.group_by("year")
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort(
        "jumlah_baris",
        descending=True
    )
    .head(20)
    .collect(engine="streaming")
)

display(distribusi_tahun)


year,jumlah_baris
str,u32
"""2016""",117505
"""2015""",88739
"""2014""",76857
"""2011""",67646
"""2013""",66173
…,…
"""2003""",7652
"""2001""",7234
"""2000""",6497


Cell 3. Daftar lengkap kolom dan tipe data

In [6]:
# Inventarisasi seluruh kolom dataset

schema = lf.collect_schema()
schema_items = list(schema.items())

schema_df = pl.DataFrame({
    "kolom": [item[0] for item in schema_items],
    "tipe_data": [str(item[1]) for item in schema_items],
})

print("Jumlah kolom:", len(schema_items))
display(schema_df)

Jumlah kolom: 41


kolom,tipe_data
str,str
"""global_id""","""Int64"""
"""local_id""","""Int64"""
"""regulation_type""","""String"""
"""enacting_body""","""String"""
"""regulation_number""","""String"""
…,…
"""kg_sanctions_json""","""String"""
"""kg_sanction_types_json""","""String"""
"""kg_concept_vector_json""","""String"""


Cell 4. Validasi tahun regulasi

In [7]:
year_num = pl.col("year").cast(pl.Int32, strict=False)

year_audit = lf.select([
    pl.len().alias("total_baris"),
    pl.col("year").n_unique().alias("jumlah_tahun_berbeda"),
    year_num.min().alias("tahun_min"),
    year_num.max().alias("tahun_max"),
    year_num.is_null().sum().alias("tahun_gagal_parse"),
    (year_num < 1800).fill_null(False).sum().alias("tahun_di_bawah_1800"),
    (year_num > 2026).fill_null(False).sum().alias("tahun_setelah_2026"),
]).collect(engine="streaming")

display(year_audit)

total_baris,jumlah_tahun_berbeda,tahun_min,tahun_max,tahun_gagal_parse,tahun_di_bawah_1800,tahun_setelah_2026
u32,u32,i32,i32,u32,u32,u32
748558,77,0,2030,0,567,2


Cell 5. Pemeriksaan keunikan global_id

In [8]:
id_audit = lf.select([
    pl.len().alias("total_baris"),
    pl.col("global_id").n_unique().alias("jumlah_global_id_unik"),
    pl.col("global_id").is_null().sum().alias("global_id_null"),
    pl.col("global_id").is_duplicated()
      .sum()
      .alias("baris_dengan_global_id_berulang"),
]).collect(engine="streaming")

display(id_audit)

total_baris,jumlah_global_id_unik,global_id_null,baris_dengan_global_id_berulang
u32,u32,u32,u32
748558,748558,0,0


Cell 6. Distribusi jenis peraturan

In [9]:
distribusi_jenis = (
    lf.group_by("regulation_type")
    .agg(pl.len().alias("jumlah_baris"))
    .sort("jumlah_baris", descending=True)
    .head(15)
    .collect(engine="streaming")
)

display(distribusi_jenis)

regulation_type,jumlah_baris
str,u32
"""PERATURAN DAERAH KABUPATEN""",261899
"""PERATURAN BUPATI""",109002
"""Unknown""",79764
"""PERATURAN DAERAH KOTA""",71978
"""PERATURAN PEMERINTAH""",46280
…,…
"""KEPUTUSAN PRESIDEN""",16133
"""PERATURAN DAERAH""",12876
"""UNDANG UNDANG""",432


Cell 7. Distribusi lembaga penerbit

In [10]:
distribusi_lembaga = (
    lf.group_by("enacting_body")
    .agg(pl.len().alias("jumlah_baris"))
    .sort("jumlah_baris", descending=True)
    .head(15)
    .collect(engine="streaming")
)

display(distribusi_lembaga)

enacting_body,jumlah_baris
str,u32
"""REPUBLIK INDONESIA""",114180
"""MA""",81218
"""Unknown""",64521
"""RI""",64362
"""BUPATI""",33404
…,…
"""PONTIANAK""",4994
"""GORONTALO""",4930
"""TRENGGALEK""",4784


Cell 8. Statistik panjang teks

In [11]:
content_len = pl.col("content").str.len_chars()
about_len = pl.col("about").str.len_chars()

content_length_stats = lf.select([
    pl.len().alias("total_baris"),

    content_len.min().alias("content_min"),
    content_len.quantile(0.25).alias("content_q25"),
    content_len.median().alias("content_median"),
    content_len.mean().alias("content_mean"),
    content_len.quantile(0.75).alias("content_q75"),
    content_len.max().alias("content_max"),

    about_len.min().alias("about_min"),
    about_len.median().alias("about_median"),
    about_len.max().alias("about_max"),
]).collect(engine="streaming")

display(content_length_stats)


total_baris,content_min,content_q25,content_median,content_mean,content_q75,content_max,about_min,about_median,about_max
u32,u32,f64,f64,f64,f64,u32,u32,f64,u32
748558,3,188.0,421.0,1234.325748,918.0,32767,1,67.0,32767


Cell 9. Kandidat outlier panjang teks

In [12]:
q1 = float(content_length_stats["content_q25"][0])
q3 = float(content_length_stats["content_q75"][0])

iqr = q3 - q1
batas_outlier = q3 + (1.5 * iqr)

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Batas atas outlier:", batas_outlier)

outlier_audit = lf.select([
    (
        pl.col("content").str.len_chars() > batas_outlier
    ).sum().alias("jumlah_kandidat_outlier")
]).collect(engine="streaming")

display(outlier_audit)

Q1: 188.0
Q3: 918.0
IQR: 730.0
Batas atas outlier: 2013.0


jumlah_kandidat_outlier
u32
89877


Cell 10. Memeriksa format tanggal

In [13]:
contoh_tanggal = (
    lf.select("effective_date")
    .unique()
    .head(30)
    .collect(engine="streaming")
)

display(contoh_tanggal)

effective_date
str
"""27 Desember 2017"""
"""9 September 2009"""
"""17 Desember 2009"""
"""6 November 2015"""
"""14 Juni 2009"""
…
"""18 Desember 2014"""
"""6 Februari 2013"""
"""4 Desember 2009"""


Cell 11. Memeriksa tahun yang tidak lazim

In [14]:
# Memeriksa distribusi nilai tahun yang tidak lazim

tahun_anomali = (
    lf.filter(
        (year_num < 1800) | (year_num > 2026)
    )
    .group_by("year")
    .agg(
        pl.len().alias("jumlah_baris")
    )
    .sort(
        "jumlah_baris",
        descending=True
    )
    .collect(engine="streaming")
)

display(tahun_anomali)

year,jumlah_baris
str,u32
"""0000""",567
"""2030""",2


Cell 12. Menghitung label Unknown

In [15]:
# Memeriksa label Unknown pada kolom utama

unknown_audit = lf.select([
    (
        pl.col("regulation_type") == "Unknown"
    ).sum().alias("regulation_type_unknown"),

    (
        pl.col("enacting_body") == "Unknown"
    ).sum().alias("enacting_body_unknown"),
]).collect(engine="streaming")

display(unknown_audit)

regulation_type_unknown,enacting_body_unknown
u32,u32
79764,64521


Cell 13. Memeriksa teks dengan panjang ekstrem

In [16]:
# Mengambil contoh teks dengan panjang minimal 32.000 karakter

teks_ekstrem = (
    lf.with_columns([
        pl.col("content")
        .str.len_chars()
        .alias("content_chars"),

        pl.col("about")
        .str.len_chars()
        .alias("about_chars"),
    ])
    .filter(
        (pl.col("content_chars") >= 32000)
        | (pl.col("about_chars") >= 32000)
    )
    .select([
        "global_id",
        "regulation_type",
        "year",
        "content_chars",
        "about_chars",
        "about",
        "content",
    ])
    .head(5)
    .collect(engine="streaming")
)

display(teks_ekstrem)

global_id,regulation_type,year,content_chars,about_chars,about,content
i64,str,str,u32,u32,str,str
246,"""PERATURAN PEMERINTAH""","""2009""",32767,96,"""JENIS DAN TARIF ATAS JENIS PEN…","""Lampiran yang merupakan bagian…"
530,"""PERATURAN PEMERINTAH""","""2009""",32767,103,"""JENIS DAN TARIF ATAS JENIS PEN…","""Lampiran Peraturan Pemerintah …"
628,"""PERATURAN PEMERINTAH""","""2009""",32767,112,"""JENIS DAN TARIF ATAS JENIS PEN…","""Lampiran Peraturan Pemerintah …"
859,"""PERATURAN PEMERINTAH""","""2008""",32767,115,"""JENIS DAN TARIF ATAS JENIS PEN…","""Lampiran Peraturan Pemerintah …"
1241,"""PERATURAN PEMERINTAH""","""2007""",32767,27,"""ORGANISASI PERANGKAT DAERAH""","""lampiran sebagai bagian yang t…"


Cell 14, periksa apakah teks terpotong

In [17]:
# Cell 14: Audit teks dengan panjang tepat 32.767 karakter

batas_teks = 32767

audit_batas_teks = lf.select([
    pl.len().alias("total_baris"),

    (
        pl.col("content").str.len_chars() == batas_teks
    ).sum().alias("content_tepat_32767"),

    (
        pl.col("about").str.len_chars() == batas_teks
    ).sum().alias("about_tepat_32767"),

    (
        pl.col("content").str.len_chars() > batas_teks
    ).sum().alias("content_di_atas_32767"),
]).collect(engine="streaming")

display(audit_batas_teks)

# Menampilkan bagian akhir dari lima contoh teks ekstrem
contoh_akhir = (
    lf.filter(
        pl.col("content").str.len_chars() == batas_teks
    )
    .select([
        "global_id",
        "regulation_type",
        "year",
        pl.col("content")
          .str.slice(-300, 300)
          .alias("akhir_content"),
    ])
    .head(5)
    .collect(engine="streaming")
)

display(contoh_akhir)

total_baris,content_tepat_32767,about_tepat_32767,content_di_atas_32767
u32,u32,u32,u32
748558,3615,99,0


global_id,regulation_type,year,akhir_content
i64,str,str,str
246,"""PERATURAN PEMERINTAH""","""2009""","""an Perorangan Lainnya, serta J…"
530,"""PERATURAN PEMERINTAH""","""2009""",""",00 | | | | Per permohonan/ Pe…"
628,"""PERATURAN PEMERINTAH""","""2009""","""r tahun | Rp | 11.891.000,00 |…"
859,"""PERATURAN PEMERINTAH""","""2008""","""ri Per orang/hari Per orang/ha…"
1241,"""PERATURAN PEMERINTAH""","""2007""","""kup jelas. Huruf k Cukup jel…"
